# Skin Cancer Detection + Generative AI Medical Assistant

Step 1: Mount Google Drive


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Step 2: Install the required libraries

In [2]:
!pip install tensorflow pandas numpy matplotlib seaborn scikit-learn pillow -q

Step 3: Import the required libraries

In [3]:
import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

Step 4: Set the dataset path

In [4]:
metadata = pd.read_csv(
    "/content/drive/MyDrive/HAM10000_metadata.csv"
)

image_dir1 = "/content/drive/MyDrive/HAM10000_images_part_1"
image_dir2 = "/content/drive/MyDrive/HAM10000_images_part_2"

print(metadata.head())
print(metadata.shape)

     lesion_id      image_id   dx dx_type   age   sex localization
0  HAM_0000118  ISIC_0027419  bkl   histo  80.0  male        scalp
1  HAM_0000118  ISIC_0025030  bkl   histo  80.0  male        scalp
2  HAM_0002730  ISIC_0026769  bkl   histo  80.0  male        scalp
3  HAM_0002730  ISIC_0025661  bkl   histo  80.0  male        scalp
4  HAM_0001466  ISIC_0031633  bkl   histo  75.0  male          ear
(10015, 7)


Step 5: Create the image paths

In [5]:
image_paths = {}

for folder in [image_dir1, image_dir2]:
    for image in os.listdir(folder):
        image_id = image.split('.')[0]
        image_paths[image_id] = os.path.join(folder, image)

metadata['path'] = metadata['image_id'].map(image_paths.get)

print(metadata[['image_id', 'path']].head())

       image_id                                               path
0  ISIC_0027419  /content/drive/MyDrive/HAM10000_images_part_1/...
1  ISIC_0025030  /content/drive/MyDrive/HAM10000_images_part_1/...
2  ISIC_0026769  /content/drive/MyDrive/HAM10000_images_part_1/...
3  ISIC_0025661  /content/drive/MyDrive/HAM10000_images_part_1/...
4  ISIC_0031633  /content/drive/MyDrive/HAM10000_images_part_2/...


Step 6: Encode the labels

In [6]:
from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()

metadata['label'] = encoder.fit_transform(metadata['dx'])

print(encoder.classes_)
print(metadata[['dx', 'label']].head())

['akiec' 'bcc' 'bkl' 'df' 'mel' 'nv' 'vasc']
    dx  label
0  bkl      2
1  bkl      2
2  bkl      2
3  bkl      2
4  bkl      2


Step 7: Load the trained model

In [8]:
from tensorflow.keras.models import load_model

cnn_model = load_model(
    "/content/drive/MyDrive/skin_cancer_cnn_model.h5"
)

print("Model loaded successfully")

Model loaded successfully


Step 8: Install Gemini AI

In [9]:
!pip install google-generativeai -q

Step 9: Set up the Gemini API

In [10]:
import google.generativeai as genai

API_KEY = "Enter Your API"

genai.configure(api_key=API_KEY)

gemini = genai.GenerativeModel("gemini-2.5-flash")

print("Gemini Connected")

Gemini Connected


/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


Step 10: Define the disease labels

In [11]:
disease_names = {
    0: "Actinic keratoses",
    1: "Basal cell carcinoma",
    2: "Benign keratosis",
    3: "Dermatofibroma",
    4: "Melanoma",
    5: "Melanocytic nevi",
    6: "Vascular lesions"
}

Step 11: Create the prediction function

In [23]:
import cv2
import numpy as np

def predict_skin(image_path):

    img = cv2.imread(image_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # Tumhare CNN model ka actual input size
    img = cv2.resize(img, (100,100))

    img = img/255.0
    img = np.expand_dims(img, axis=0)

    prediction = cnn_model.predict(img, verbose=0)

    predicted = np.argmax(prediction)
    confidence = np.max(prediction)

    return disease_names[predicted], confidence

Step 12: Create the AI medical report function

In [24]:
def generate_medical_report(disease):

    prompt = f"""
    Explain {disease} skin disease.

    Include:
    1. What is this disease?
    2. Symptoms
    3. Causes
    4. Treatment options
    5. Prevention methods
    6. When should a patient consult a doctor?

    Use simple language.
    """

    response = gemini.generate_content(prompt)

    return response.text

Step 13: Test the AI assistant

In [25]:
report = generate_medical_report("Melanoma")

print(report)

Melanoma is the most serious type of skin cancer. It develops in the cells that produce melanin – the pigment that gives your skin its color. While it's less common than some other types of skin cancer, it's more dangerous because it has a higher chance of spreading to other parts of the body if not caught and treated early.

Let's break it down:

---

### 1. What is this disease?

Melanoma is a type of cancer that starts in cells called **melanocytes**. These cells are responsible for making melanin, which protects your skin from the sun's UV rays and gives it its tan or brown color. When these melanocytes grow abnormally and uncontrollably, they can form a melanoma. It can appear on any skin surface, even areas not exposed to the sun, but it's most common on the face, trunk, arms, and legs. It can also develop in existing moles or appear as a new, unusual-looking spot on the skin.

### 2. Symptoms

The main symptom of melanoma is a **change in an existing mole or the development of a

Step 14: Select a test image from Google Drive

In [26]:
import os

images = os.listdir('/content/drive/MyDrive/HAM10000_images_part_1')

print(images[:10])

['ISIC_0028360.jpg', 'ISIC_0028303.jpg', 'ISIC_0028329.jpg', 'ISIC_0028358.jpg', 'ISIC_0028327.jpg', 'ISIC_0028348.jpg', 'ISIC_0028317.jpg', 'ISIC_0028308.jpg', 'ISIC_0028339.jpg', 'ISIC_0028319.jpg']


In [27]:
image_path = "/content/drive/MyDrive/HAM10000_images_part_1/ISIC_0024306.jpg"

disease, confidence = predict_skin(image_path)

print("Disease:", disease)
print("Confidence:", confidence)

Disease: Melanocytic nevi
Confidence: 0.9999944


Step 15: Generate the generative AI medical report

In [30]:
def generate_medical_report(disease):

    prompt = f"""
    You are a professional medical AI assistant.

    Explain the skin disease: {disease}

    Include:
    1. What is this disease?
    2. Common symptoms
    3. Possible causes
    4. Treatment options
    5. Prevention methods
    6. When should a patient consult a doctor?

    Use simple and easy English.
    """

    response = gemini.generate_content(prompt)

    return response.text

Step 16: Generate the report

In [31]:
report = generate_medical_report(disease)

print(report)

Let's talk about **Melanocytic Nevi**, which are more commonly known as **moles**.

---

### 1. What is this disease?

Melanocytic nevi, or moles, are very common, usually harmless growths on the skin. They happen when special skin cells called **melanocytes** (which make the pigment that gives our skin its color) grow in clusters instead of being spread out. Most people have between 10 and 40 moles. They can be present at birth (congenital moles) or develop later in life (acquired moles).

### 2. Common symptoms

Moles usually have these common features:

*   **Size:** They are typically small, often less than 6 millimeters (about the size of a pencil eraser).
*   **Shape:** Most are round or oval.
*   **Color:** They can be tan, brown, black, red, pink, blue, or even skin-colored.
*   **Surface:** They can be flat and smooth, or raised and dome-shaped. Some can even have hair growing from them.
*   **Borders:** They usually have clear, regular borders.
*   **Feel:** Moles typically d

Step 17: Display the final output

In [32]:
print("="*50)
print("AI SKIN CANCER ASSISTANT")
print("="*50)

print("\nPredicted Disease:", disease)
print("Confidence:", confidence)

print("\nMedical Report:\n")
print(report)

AI SKIN CANCER ASSISTANT

Predicted Disease: Melanocytic nevi
Confidence: 0.9999944

Medical Report:

Let's talk about **Melanocytic Nevi**, which are more commonly known as **moles**.

---

### 1. What is this disease?

Melanocytic nevi, or moles, are very common, usually harmless growths on the skin. They happen when special skin cells called **melanocytes** (which make the pigment that gives our skin its color) grow in clusters instead of being spread out. Most people have between 10 and 40 moles. They can be present at birth (congenital moles) or develop later in life (acquired moles).

### 2. Common symptoms

Moles usually have these common features:

*   **Size:** They are typically small, often less than 6 millimeters (about the size of a pencil eraser).
*   **Shape:** Most are round or oval.
*   **Color:** They can be tan, brown, black, red, pink, blue, or even skin-colored.
*   **Surface:** They can be flat and smooth, or raised and dome-shaped. Some can even have hair growing

Step 18: Generate the PDF report

In [33]:
!pip install fpdf -q

  Preparing metadata (setup.py) ... done


In [34]:
report = generate_medical_report(disease)
print(report)

As a professional medical AI assistant, I can explain Melanocytic Nevi (common moles) for you in simple and easy English.

---

### Melanocytic Nevi: Understanding Common Moles

**1. What is this disease?**
Melanocytic nevi, more commonly known as **moles**, are very common skin growths. They are basically clusters of pigment-producing cells called melanocytes. These cells are responsible for giving your skin its color. Moles are almost always **benign (non-cancerous)**, meaning they are harmless. Most people have several moles on their body, and they can appear anywhere on the skin.

**2. Common symptoms**
Moles can vary quite a bit, but here are their typical features:
*   **Appearance:** They are usually small spots on the skin.
*   **Shape:** Most are round or oval.
*   **Color:** They can be various shades of brown, black, tan, pink, or even skin-colored.
*   **Texture:** They can be flat or slightly raised (bumpy). Some may have hair growing from them.
*   **Borders:** Their edge

Step 18: Install the PDF library

In [35]:
!pip install fpdf -q

Step 19: Create the PDF generation function

In [36]:
from fpdf import FPDF

def create_pdf_report(disease, confidence, report):

    pdf = FPDF()

    pdf.add_page()

    pdf.set_font("Arial", "B", 16)
    pdf.cell(200, 10, "AI Skin Cancer Medical Report", ln=True)

    pdf.ln(5)

    pdf.set_font("Arial", "", 12)

    pdf.cell(200, 10,
             f"Predicted Disease: {disease}",
             ln=True)

    pdf.cell(200, 10,
             f"Confidence: {confidence:.4f}",
             ln=True)

    pdf.ln(5)

    report = report.encode(
        "latin-1",
        "replace"
    ).decode("latin-1")

    pdf.multi_cell(
        0,
        8,
        report
    )

    pdf.output(
        "skin_cancer_report.pdf"
    )

    print("PDF Created Successfully")

Step 20: Create the PDF

In [37]:
create_pdf_report(
    disease,
    confidence,
    report
)

PDF Created Successfully


Step 21: Download the PDF

In [38]:
from google.colab import files

files.download(
    "skin_cancer_report.pdf"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>